# Метод 1. Скрытое API сайта Минфина и таблицы из docx

Задача была собрать все купонные выплаты и погашения ОФЗ за 2024-2026 годы в одну таблицу: дата выплаты, номер выпуска, тип платежа, сумма. Источник один: раздел Минфина «Информация о выплаченных купонных доходах и погашении», где на каждую дату выплаты публикуется сообщение с таблицей.

Начинать я хотел с самого простого. Готовых данных не нашлось: архив Банка России в нужном формате обновлялся только до конца 2012 года. Официального API у Минфина для этого раздела нет. У Московской биржи API есть, и по выплате 07.10.2026 он показал те же выпуски и тот же купон, но только на одну облигацию. Объем выпуска на бирже не равен числу облигаций в обращении (у 26245 на бирже 1,25 трлн руб., а по сумме Минфина в обращении около 1,04 трлн), поэтому суммы на выпуск оттуда не получить. Оставалось искать запросы, которыми сайт сам подгружает данные. Эта тетрадь про них.

Таблицу сообщения здесь я беру из docx, который приложен к каждому сообщению. В двух других тетрадях та же таблица берется из HTML-страницы и из браузера, и совпадение трех результатов в `04_compare` служит главной проверкой: ошибка разбора в одном методе проявилась бы расхождением.

## 0. Период

Даты выплаты, за которые собираются данные, в формате ДД.ММ.ГГГГ, обе включительно. Пояснения и цифры ниже относятся к периоду 01.01.2024-27.10.2026.

In [4]:
# период по дате выплаты, обе даты включительно, формат ДД.ММ.ГГГГ
DATE_FROM = "01.01.2024"
DATE_TO = "27.10.2026"

## 1. Настройки: сайт пускает только подпись браузера

Про то, что сайт смотрит на подпись запроса, я узнал еще на тестовых запросах, которые не вошли в тетрадь. Запрос с подписью requests по умолчанию получил 503 «Доступ к сайту временно ограничен владельцем веб-ресурса», честная подпись с названием проекта и ссылкой на репозиторий тоже, а подпись Chrome прошла. Поэтому здесь подпись браузера, запросы идут по одному с паузой в секунду, а после 503 тетрадь сразу останавливается: повторяется только обрыв связи.

Внутренний адрес `/moduleRouter/` на первую же попытку ответил «ModuleRouter require XmlHttpRequest!»: он принимает только фоновые запросы страницы. Поэтому запросы к нему помечены заголовком `X-Requested-With`, как это делает сама страница.

Фильтр сайта работает по дате публикации, а мне нужна дата выплаты. Сообщение публикуют в день выплаты или позже, поэтому список я беру с запасом, а период `PERIOD` отрезаю уже по дате выплаты. Все скачанное ложится в `raw/`, чтобы при перезапуске не ходить на сайт заново.

In [5]:
%pip install -r ../requirements.txt

   ---------------------------------------- 0.0/10.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/10.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/10.9 MB ? eta -:--:--
    --------------------------------------- 0.3/10.9 MB ? eta -:--:--
   - -------------------------------------- 0.5/10.9 MB 1.4 MB/s eta 0:00:08
   ---- ----------------------------------- 1.3/10.9 MB 2.4 MB/s eta 0:00:05
   ---------- ----------------------------- 2.9/10.9 MB 4.0 MB/s eta 0:00:03
   ----------------------- ---------------- 6.3/10.9 MB 6.9 MB/s eta 0:00:01
   --------------------------- ------------ 7.6/10.9 MB 6.8 MB/s eta 0:00:01
   ---------------------------------------- 10.9/10.9 MB 9.1 MB/s  0:00:01
   ---------------------------------------- 0.0/38.8 MB ? eta -:--:--
   --- ------------------------------------ 3.7/38.8 MB 25.1 MB/s eta 0:00:02
   ------- -------------------------------- 6.8/38.8 MB 16.4 MB/s eta 0:00:02
   ----------- -------------------


[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
import io
import re
import time
from datetime import datetime, timedelta
from pathlib import Path

import docx
import pandas as pd
import requests
from bs4 import BeautifulSoup

BASE = "https://minfin.gov.ru"
ROUTER = f"{BASE}/moduleRouter/"
# подпись браузера: на подпись requests по умолчанию сайт отвечает 503
UA = ("Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
      "(KHTML, like Gecko) Chrome/141.0.0.0 Safari/537.36")
# итоговый период по дате выплаты из ячейки выше; сайт фильтрует ленту по дате публикации,
# а сообщение публикуют в день выплаты или позже (бывало через неделю), поэтому конец берется с запасом
start, end = datetime.strptime(DATE_FROM, "%d.%m.%Y"), datetime.strptime(DATE_TO, "%d.%m.%Y")
PERIOD = (f"{start:%Y-%m-%d}", f"{end:%Y-%m-%d}")
PUBLISHED_FROM, PUBLISHED_TO = f"{start:%d.%m.%Y}", f"{end + timedelta(days=30):%d.%m.%Y}"
PAUSE = 1  # секунд между запросами
# сюда кладется все скачанное; при перезапуске оно не качается заново
RAW = Path("raw")
RAW.mkdir(exist_ok=True)
XHR = {"X-Requested-With": "XMLHttpRequest"}  # без него /moduleRouter/ отвечает отказом

session = requests.Session()
session.headers.update({"User-Agent": UA, "Accept-Language": "ru-RU,ru;q=0.9"})


def request(method, url, **kwargs):
    for attempt in range(3):  # повторяем только обрыв соединения, ответ сервера не повторяем
        try:
            resp = session.request(method, url, timeout=(15, 60), **kwargs)
            break
        except requests.ConnectionError:
            if attempt == 2:
                raise
            time.sleep(5)
    resp.raise_for_status()  # 503 от защиты сайта сразу останавливает тетрадь, без повторов
    time.sleep(PAUSE)
    return resp

## 2. JSON-поиск сайта отдает не больше четырех документов

Первым кандидатом на API был поиск в шапке сайта. Еще когда я разбирал JavaScript страницы, я заметил, что при наборе текста он отправляет POST на `/moduleRouter/?page_id=10102&area_id=26` и получает JSON со списком документов. Я повторил этот запрос с текстом «Информация о выплатах купонного дохода».

In [7]:
# тот же запрос, что отправляет поле поиска в шапке сайта
resp = request("POST", ROUTER, params={"page_id": 10102, "area_id": 26}, headers=XHR,
               files={"q_26": (None, "Информация о выплатах купонного дохода")})
# документы лежат в первом разделе ответа
section = resp.json()["data"][0]
print(section["sectionName"], "| документов в ответе:", len(section["list"]),
      "| есть еще:", section["hasMore"])
pd.DataFrame(section["list"])[["date", "title", "url"]]

Документы | документов в ответе: 4 | есть еще: True


,date,title,url
0,2026-10-28T09:26:00+03:00,Информация о выплатах купонного дохода 28 октя...,/ru/perfomance/public_debt/internal/operations...
1,2026-10-22T10:49:00+03:00,Информация о выплатах купонного дохода 22 октя...,/ru/perfomance/public_debt/internal/operations...
2,2026-10-21T09:37:00+03:00,Информация о выплатах купонного дохода 21 октя...,/ru/perfomance/public_debt/internal/operations...
3,2026-10-14T09:39:00+03:00,Информация о выплате купонного дохода 14 октяб...,/ru/perfomance/public_debt/internal/operations...


JSON пришел, но документов в нем всего 4 и стоит флаг `hasMore`: есть еще. Способа попросить следующие в коде страницы нет, так что полный список этим путем не собрать. Найти одно сообщение по точной дате так можно, но мне нужны все сообщения за три года.

## 3. Полный список дает внутренний адрес ленты

Тогда я посмотрел, как подгружает карточки сама лента. Кнопка «Посмотреть еще» и фильтр по дате работают через тот же `/moduleRouter/`, только с `area_id=65`: в ответ приходит один блок карточек, без шапки и меню. Фильтр и номер страницы передаются параметрами `P_DATE_from_65`, `P_DATE_to_65` и `page_65`, а число страниц написано на самой кнопке. Из карточки мне нужны номер сообщения, заголовок с датой выплаты, дата публикации и ссылка на docx.

In [8]:
# внутренний адрес ленты: только блок карточек, с фильтром по дате публикации
def list_page(page):
    resp = request("GET", ROUTER, headers=XHR, params={
        "page_id": 10102, "area_id": 65, "page_65": page,
        "P_DATE_from_65": PUBLISHED_FROM, "P_DATE_to_65": PUBLISHED_TO})
    return BeautifulSoup(resp.text, "lxml")


# из карточки: номер сообщения, заголовок с датой выплаты, дата публикации, ссылка на docx
def parse_cards(soup):
    cards = []
    for card in soup.select("div.document_card"):
        title = card.select_one("a.document_title")
        file = card.select_one("a.file_item")  # бывает, что docx к сообщению еще не приложен
        cards.append({"doc_id": re.search(r"id_65=(\d+)", title["href"])[1],
                      "title": title["title"],
                      "published_date": pd.to_datetime(
                          card.select_one(".document_footer .date").get_text(strip=True), dayfirst=True),
                      "docx_url": BASE + file["href"] if file else None})
    return cards


first = list_page(1)
# число страниц написано на кнопке «Посмотреть еще»
more = first.select_one("a.button_more")  # кнопки нет, если период умещается на одну страницу
n_pages = int(more["data-page-count"]) if more else 1
soups = [first] + [list_page(page) for page in range(2, n_pages + 1)]
cards = pd.DataFrame([card for soup in soups for card in parse_cards(soup)])
print("На сайте:", first.select_one(".count_value").get_text(strip=True), "документов,",
      n_pages, "страниц | собрано карточек:", len(cards))
cards.head()

На сайте: 141 документов, 16 страниц | собрано карточек: 141


,doc_id,title,published_date,docx_url
0,317861,Информация о выплатах купонного дохода 28 октя...,2026-10-28,NaN
1,317815,Информация о выплатах купонного дохода 22 октя...,2026-10-22,https://minfin.gov.ru/common/upload/library/20...
2,317799,Информация о выплатах купонного дохода 21 октя...,2026-10-21,https://minfin.gov.ru/common/upload/library/20...
3,317739,Информация о выплате купонного дохода 14 октяб...,2026-10-14,https://minfin.gov.ru/common/upload/library/20...
4,317695,Информация о выплатах купонного дохода и погаш...,2026-10-07,https://minfin.gov.ru/common/upload/library/20...


Сайт насчитал 141 документ за интервал публикаций с 01.01.2024 по 26.11.2026, это 16 страниц, и собрались все карточки: список полный. Сообщение о выплате 28.10.2026 в период не входит и отпадет, когда я отрежу период по дате выплаты.

## 4. Тип платежа в таблице задает строка-раздел

Прежде чем качать файлы, я разобрался, как устроена таблица сообщения. Вот сообщение за 07.10.2026, показаны не все строки:

| № выпуска | № купона | Ставка, % | Купонный доход, руб. | Сумма, млн руб. |
|---|---|---|---|---|
| **Выплаты купонного дохода** | | | | |
| 26226RMFS | 16 | 7,95 | 39,64 | 14 556,237 |
| 26232RMFS | 14 | 6,00 | 29,92 | 13 402,177 |
| **Погашение** | | | | |
| 26226RMFS | | | | 367 210,831 |
| **Итого перечислено из бюджета, млн руб.** | | | | 473 585,317 |

Тип платежа не записан в строке: его задает строка-раздел над ней, поэтому `parse_table` идет сверху вниз и помнит текущий раздел. Номер выпуска стоит в первой ячейке, сумма в последней, а строку «Итого» функция возвращает отдельно, для проверки. «Купонный доход» указан на одну облигацию, а «Сумма» на весь выпуск: у 26226RMFS 14 556,237 млн руб. / 39,64 руб. дает 367,2 млн облигаций, и это почти ровно сумма его погашения в тот же день.

Правило «незнакомая строка останавливает разбор» я поставил сознательно: лучше упасть, чем молча потерять платеж. При первом полном прогоне оно сработало дважды. В 43 из 143 сообщений раздел назывался «Выплата купонного дохода», в единственном числе, и это я просто добавил в разбор. А у выпусков 46011, 46012 и 46023 нашелся раздел «Частичное погашение»: это ОФЗ-АД, где долг гасится частями. Сначала я записывал его как погашение с пометкой, потом выделил в отдельный тип, чтобы его нельзя было принять за полное погашение выпуска.

Числа на сайте записаны как `367 210,831`, иногда с неразрывным пробелом, а дата выплаты стоит в заголовке сообщения словами, с месяцем в родительном падеже. Ячейка ниже только определяет функции, к сайту не обращается.

In [9]:
# в заголовках месяц стоит в родительном падеже: «7 октября 2026 г.»
MONTHS = {m: i for i, m in enumerate(
    ["января", "февраля", "марта", "апреля", "мая", "июня", "июля",
     "августа", "сентября", "октября", "ноября", "декабря"], start=1)}


def payment_date(title):
    """Дата выплаты из заголовка: «... 7 октября 2026 г.» -> 2026-10-07."""
    day, month, year = re.search(r"(\d{1,2})\s+([а-я]+)\s+(\d{4})", title).groups()
    return pd.Timestamp(int(year), MONTHS[month], int(day))


# «367 210,831» -> 367210.831; между разрядами бывает неразрывный пробел
def to_number(text):
    return float(text.replace("\xa0", "").replace(" ", "").replace(",", "."))


def parse_table(rows):
    """Строки таблицы (списки текстов ячеек) -> записи о платежах и строка «Итого».

    Тип платежа задают строки-разделы: «Выплаты (Выплата) купонного дохода» -> купон,
    «Погашение» -> погашение, «Частичное погашение» (амортизация ОФЗ-АД) -> частичное погашение.
    Любая незнакомая строка останавливает разбор: так данные не теряются молча.
    """
    records, total, kind = [], None, None
    for cells in rows:
        cells = [c.strip() for c in cells]
        first = cells[0] if cells else ""
        if not any(cells) or first.lower().startswith("№ выпуска"):
            continue
        if re.match(r"Выплат[аы] купонного дохода", first):  # бывает и «Выплата», и «Выплаты»
            kind = "купон"
        elif first.startswith("Погашение"):
            kind = "погашение"
        elif first.startswith("Частичное погашение"):
            kind = "частичное погашение"
        elif first.startswith("Итого"):
            total = to_number(cells[-1])
        elif re.fullmatch(r"\d{5}RMFS", first) and kind == "купон":
            records.append({"issue": first, "payment_type": kind,
                            "coupon_no": int(cells[1]), "coupon_rate_pct": to_number(cells[2]),
                            "coupon_per_bond_rub": to_number(cells[3]),
                            "amount_mln_rub": to_number(cells[4])})
        elif re.fullmatch(r"\d{5}RMFS", first) and kind in ("погашение", "частичное погашение"):
            records.append({"issue": first, "payment_type": kind,
                            "amount_mln_rub": to_number(cells[-1])})
        else:
            raise ValueError(f"Неожиданная строка таблицы: {cells}")
    return records, total

## 5. Объединенные ячейки docx не мешают разбору

python-docx отдает строки таблицы списками ячеек, но объединенную ячейку повторяет в каждой клетке: строка раздела выглядит как четыре одинаковые ячейки и пустая пятая. Правилу «номер из первой ячейки, сумма из последней» это не мешает, так что для docx ничего особенного не понадобилось. К каждой записи я добавлял номер сообщения, дату выплаты, дату публикации и ссылку на файл.

In [10]:
def load_docx(card):
    path = RAW / f"{card.doc_id}.docx"
    if not path.exists():  # кэш: при повторном запуске файл не качаем
        path.write_bytes(request("GET", card.docx_url).content)
    return docx.Document(path)


# без docx методу не из чего взять таблицу: такие сообщения откладываются и печатаются списком,
# а если они попали в период, это покажет сверка с другими методами
no_docx = cards[cards["docx_url"].isna()]
if len(no_docx):
    print("Без docx, пропущены:", "; ".join(f"{c.doc_id} {c.title}" for c in no_docx.itertuples()))

# totals хранит «Итого» каждого сообщения для проверки
records, totals = [], []
for card in cards.dropna(subset=["docx_url"]).itertuples():
    document = load_docx(card)
    # объединенные ячейки docx повторяют текст, parse_table на это рассчитан
    rows = [[cell.text for cell in row.cells] for table in document.tables for row in table.rows]
    found, total = parse_table(rows)
    for record in found:
        record.update(doc_id=card.doc_id, payment_date=payment_date(card.title),
                      published_date=card.published_date, source_url=card.docx_url)
    records += found
    totals.append({"doc_id": card.doc_id, "total": total})
print("Разобрано сообщений:", len(totals), "| строк:", len(records))

Без docx, пропущены: 317861 Информация о выплатах купонного дохода 28 октября 2026 г.
Разобрано сообщений: 140 | строк: 449


К сообщению о выплате 28.10.2026 Минфин на момент сбора еще не приложил docx, и метод его пропустил: в период оно все равно не входит, а если бы входило, это показала бы сверка с другими методами. Остальные 140 сообщений разобрались, получилось 449 строк, и разбор нигде не остановился: незнакомых строк в таблицах нет.

## 6. Платежи складываются в одну длинную таблицу

Сначала я думал соединять таблицы сообщений через join, но это неверно: сообщения за разные даты имеют одинаковые колонки и описывают разные события, их нужно ставить друг под другом. Купон и погашение одного выпуска в один день тоже остаются двумя строками с разным типом, и получается панель «дата × выпуск × тип → сумма».

Кроме четырех обязательных колонок я оставил номер купона, ставку, купон на облигацию и ссылки на источник: они нужны для проверок и чтобы любую строку можно было найти на сайте. Колонку «вид ОФЗ» я пробовал определять по первым цифрам номера, но на сериях 36xxx и 39xxx это перестало работать, а в таблицах Минфина вида нет, так что от нее я отказался.

In [11]:
COLUMNS = ["payment_date", "issue", "payment_type", "amount_mln_rub", "coupon_no",
           "coupon_rate_pct", "coupon_per_bond_rub", "doc_id", "published_date",
           "source_url"]

# записи всех сообщений друг под другом: одна строка = один платеж
payments = pd.DataFrame(records)
# у погашений номера купона нет; Int64 хранит пропуск, не делая номера дробными
payments["coupon_no"] = payments["coupon_no"].astype("Int64")
# период режется по дате выплаты, а не по дате публикации
payments = (payments[payments["payment_date"].between(*PERIOD)]
            .sort_values(["payment_date", "payment_type", "issue"])
            .reset_index(drop=True)[COLUMNS])
print("Сообщений в периоде:", payments["doc_id"].nunique(), "| строк:", len(payments))
payments.head()

Сообщений в периоде: 140 | строк: 449


,payment_date,issue,payment_type,amount_mln_rub,coupon_no,coupon_rate_pct,coupon_per_bond_rub,doc_id,published_date,source_url
0,2024-01-17,26227RMFS,купон,14760.000,10,7.40,36.90,305799,2024-01-17,https://minfin.gov.ru/common/upload/library/20...
1,2024-01-17,26234RMFS,купон,11146.268,7,4.50,22.44,305799,2024-01-17,https://minfin.gov.ru/common/upload/library/20...
2,2024-01-24,24021RMFS,купон,11043.000,16,14.76,36.81,305871,2024-01-24,https://minfin.gov.ru/common/upload/library/20...
3,2024-01-24,26212RMFS,купон,12302.500,22,7.05,35.15,305871,2024-01-24,https://minfin.gov.ru/common/upload/library/20...
4,2024-01-24,29015RMFS,купон,15084.617,13,14.76,36.81,305871,2024-01-24,https://minfin.gov.ru/common/upload/library/20...


В периоде все 140 сообщений и 449 строк: сообщение о выплате 28.10.2026 метод пропустил еще раньше.

## 7. Все 140 сообщений сходятся с «Итого»

Проверок две. Первая: сумма строк каждого сообщения равна его «Итого». Допуск 0,001 млн руб. на строку я ввел, когда еще на тестовых страницах увидел сообщение за 31.01.2024, где сумма строк 112 589,114, а «Итого» 112 589,115: Минфин округляет каждую сумму отдельно. Вторая: ключ «дата + выпуск + тип» не должен повторяться.

In [12]:
# Проверка 1: в каждом сообщении сумма строк равна «Итого» (допуск на округление 0,001 млн на строку)
check = (payments.groupby("doc_id")["amount_mln_rub"].agg(["sum", "size"])
         .join(pd.DataFrame(totals).set_index("doc_id")))
check["diff"] = (check["total"] - check["sum"]).round(3)
mismatch = check[check["diff"].abs() > 0.001 * check["size"]]
print("Сообщений проверено:", len(check), "| не сходится с «Итого»:", len(mismatch))

# Проверка 2: нет повторов ключа «дата + выпуск + тип»
duplicates = payments[payments.duplicated(["payment_date", "issue", "payment_type"], keep=False)]
print("Повторов ключа:", len(duplicates))

mismatch

Сообщений проверено: 140 | не сходится с «Итого»: 0
Повторов ключа: 0


,sum,size,total,diff
doc_id,,,,


Расхождений с «Итого» 0 из 140, повторов 0. Таблица под выводом это список сообщений с расхождением, и она пуста.

## 8. Суммы по годам выглядят правдоподобно

Напоследок я посмотрел на сводку по годам, чтобы оценить результат на глаз.

In [13]:
# строки и суммы по годам и типам платежей
summary = payments.pivot_table(index=payments["payment_date"].dt.year, columns="payment_type",
                               values="amount_mln_rub", aggfunc=["count", "sum"]).round(1)
summary.index.name = "год"
summary

count                                      sum             \
payment_type купон погашение частичное погашение      купон  погашение   
год                                                                      
2024           152         8                   2  2186617.7  1441861.6   
2025           144         7                   2  3081422.3  1406539.7   
2026           129         3                   2  2890156.2  1179185.0   

                                  
payment_type частичное погашение  
год                               
2024                     29503.9  
2025                     29505.8  
2026                     29501.4

Купонов по 129-152 в год, полных погашений от 3 до 8, но каждое крупное, потому что гасится весь выпуск сразу. Частичных погашений по 2 в год, около 29,5 млрд руб. ежегодно. 2026 год неполный: период заканчивается 27.10.2026, последняя выплата в нем 22.10.2026.

## 9. Результат сохраняется для сверки

Файл `ofz_payments_api_<с>_<по>.csv` (период в имени) читает `04_compare/compare.ipynb` и сверяет его с двумя другими методами. Весь сбор этим методом занимает около 3,7 минуты.

In [14]:
# этот файл сверяет 04_compare/compare.ipynb
name = f"ofz_payments_api_{PERIOD[0]}_{PERIOD[1]}.csv"  # период в имени файла
payments.to_csv(name, index=False, encoding="utf-8")
print("Сохранено:", len(payments), "строк в", name)

Сохранено: 449 строк в ofz_payments_api_2024-01-01_2026-10-27.csv
